In [ ]:
import json
PARAMS = json.load(open("../params.json", "r"))

SEED = PARAMS["SEED"]
S_MAX = PARAMS["S_MAX"]
S_MIN = PARAMS["S_MIN"]

SAMPLING_SIZE = PARAMS["SAMPLING_SIZE"]
MAXLEN_A = PARAMS["MAXLEN_A"]

HF_TOKEN = PARAMS["HF_TOKEN"] 
MODEL_NAME = PARAMS["MODEL_NAME"] 
USERNAME = PARAMS["USERNAME"] 

ESSAY_SET = PARAMS["ESSAY_SET"]

In [ ]:
CUDA_DEVICE = "cuda:0"
import json
import pandas as pd
import numpy as np
import pickle
from transformers import BertTokenizer, BertModel
import torch
import gc
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
from huggingface_hub import login as hf_login
from huggingface_hub import HfApi
import seaborn as sns

In [ ]:
print(torch.cuda.is_available())

torch.cuda.set_device(CUDA_DEVICE)

import gc
torch.cuda.empty_cache()
gc.collect()

In [ ]:
# add names
with open("../../nllf_method/bsq_lab/output/transformed_names.json", "r") as json_file:
    nllf_transformed_names = json.load(json_file)

with open("../lf/transformed_names.json", "r") as json_file:
    ef_transformed_names = json.load(json_file)

rename = {**ef_transformed_names, **nllf_transformed_names}
rename

In [ ]:
from huggingface_hub import hf_hub_url, cached_download

repo_name = MODEL_NAME
config_file_url = hf_hub_url(f"{USERNAME}/"+repo_name, filename="cls_layer.torch")
value = cached_download(config_file_url)
cls_layer = torch.load(value).cuda()

the_model = BertModel.from_pretrained(f"{USERNAME}/"+repo_name).cuda()
the_tokenizer = BertTokenizer.from_pretrained(f"{USERNAME}/"+repo_name, do_lower_case=False)

class DatasetTaskClassification(Dataset):
    def __init__(self, df, maxlen_A=MAXLEN_A, label=True, tokenizer=None):
        self.df = df
        self.tokenizer = tokenizer
        self.maxlen_A = maxlen_A
        self.label = label
        self.wte = BertModel.from_pretrained(f"{USERNAME}/"+repo_name).cpu().embeddings.word_embeddings

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        score = 0.0
        if self.label:
            score = float(self.df.loc[index, "score"])

        a = self.df.loc[index, "essay"]
        sentence2 = str(a)
        sentence2 = "" if sentence2 == "nan" else sentence2
        sentence2 = sentence2.strip()
                
        tokens2 = self.tokenizer.tokenize(sentence2) if len(sentence2)>0 else ["[UNK]"]

        if len(tokens2) <= self.maxlen_A:
            tokens2 = tokens2 + ['[PAD]' for _ in range(self.maxlen_A - len(tokens2))]
        else:
            tokens2 = tokens2[:self.maxlen_A]
                
        tokens = ["[CLS]"]+tokens2+["[SEP]"]
        tokens_ids = self.tokenizer.convert_tokens_to_ids(tokens)
        tokens_ids_tensor = torch.tensor(tokens_ids)
        attn_mask = (tokens_ids_tensor != 1).long() # [PAD] => 1

        with torch.no_grad():
            # Get input embeddings (e.g., from the model's embedding layer)
            embedding_output = self.wte(tokens_ids_tensor)
        
        return tokens_ids_tensor, embedding_output, attn_mask, score
    
class RegressionModel(nn.Module):
    def __init__(self):
        super(RegressionModel, self).__init__()
        torch.manual_seed(SEED)
        
        self.bert_layer = the_model.cuda()
        self.cls_layer = cls_layer
        self.relu = nn.ReLU(inplace=False)

    def forward(self, input_embeds, attn_masks):

        cont_reps = self.bert_layer(inputs_embeds=input_embeds, attention_mask=attn_masks)
            
        cls_rep = cont_reps.last_hidden_state[:, 0]

        post_relu = self.relu(cls_rep)
        prelogits = self.cls_layer(post_relu)
        
        return prelogits

In [ ]:
df = pd.read_csv(f"../../../data/essay_set_{ESSAY_SET}.csv", index_col=0)
df = df[df["split"] == "test"]
indexs = df.index
df = df.reset_index()

data_set = DatasetTaskClassification(df = df, label=False, tokenizer=the_tokenizer)
data_loader = DataLoader(data_set, batch_size = 1, num_workers = 2, shuffle=False)

In [ ]:
import torch
from captum.attr import IntegratedGradients

In [ ]:
net = RegressionModel()

In [ ]:
import torch
from captum.attr import IntegratedGradients
import re
import matplotlib.pyplot as plt

# Function to split text into clauses based on punctuation
def split_into_clauses(tokens):
    # Ensure 'text' is a string (if it's a list, join into a string)
    if isinstance(tokens, list):
        first = tokens[0]
        text = the_tokenizer.convert_tokens_to_string(tokens[1:-1])
        last = tokens[-1]
    
    # This uses regular expressions to split text into clauses
    # Clauses are defined by sentences or sections separated by punctuation (., ;, !, ?)
    clauses = re.split(r'[.!?;]', text)  # Split by punctuation
    clauses = [first]+[clause.strip() for clause in clauses if clause.strip()]+[last]
    return clauses

def forward_func(input_embeds, attention_mask):
    logits = net(input_embeds, attention_mask)
    return logits

# Define function to compute attributions for each clause
def compute_clause_attributions(input_ids, seq, attn_masks, criterion=max):
    # Split the input sequence into clauses (by tokens)
    clauses = split_into_clauses(the_tokenizer.convert_ids_to_tokens(input_ids[0].cpu().numpy()))

    baseline_seq = torch.full_like(seq, the_tokenizer.pad_token_id).cuda()

    # Initialize Integrated Gradients
    ig = IntegratedGradients(forward_func)

    # Compute attributions (by tokens)
    attributions, _ = ig.attribute(inputs=seq,
                                       baselines=baseline_seq,
                                       additional_forward_args=(attn_masks,),
                                       return_convergence_delta=True)

    torch.cuda.empty_cache()  # Clear the cached memory in CUDA

    # Sum attributions across the embedding dimension to get relevance per token
    attributions = attributions.sum(dim=-1).squeeze(0)  # Sum across embedding dimensions
    attributions = attributions.detach().cpu().numpy()

    # Create a dictionary to hold total attributions by clause
    clause_attributions = {}
    token_idx = 1
    # Loop over each clause and sum the attributions for the tokens within that clause
    for clause in clauses[1:-1]:
        clause_tokens = the_tokenizer.tokenize(clause)
        clause_length = len(clause_tokens)
        clause_attr = criterion(attributions[token_idx:token_idx + clause_length])  # Sum token attributions for the clause
        clause_attributions[clause] = clause_attr
        token_idx += clause_length  # Move to the next clause

    return clause_attributions

In [ ]:
y_true = df["score"].values
pred = pd.read_csv("pred/lf/test.csv", index_col=0)
y_pred = pred["pred"]

m, M = np.min(y_true), np.max(y_true)
scaled_true = (S_MIN + ((pd.Series(y_true) - m) / (M - m)) * (S_MAX - S_MIN)).values

m, M = np.min(y_pred), np.max(y_pred)#np.percentile(y_pred, 1), np.percentile(y_pred, 99) 
scale_preds = (S_MIN + ((pd.Series(y_pred) - m) / (M - m)) * (S_MAX - S_MIN)).values

In [ ]:
df["test"] = scaled_true
df["pred"] = scale_preds

In [ ]:
with open('../../../data/question.json', 'r') as f:
    meta_type = json.load(f)

p = meta_type[str(ESSAY_SET)]["question"]
p

In [ ]:
wte = BertModel.from_pretrained(f"{USERNAME}/"+repo_name).cpu().embeddings.word_embeddings

def preprocess(df, essay_id):
    maxlen_A =MAXLEN_A
    a = df[df["essay_id"] == essay_id].iloc[0]["essay"]
    sentence2 = str(a)
    sentence2 = "" if sentence2 == "nan" else sentence2
    sentence2 = sentence2.strip()
            
    tokens2 = the_tokenizer.tokenize(sentence2) if len(sentence2)>0 else ["[UNK]"]

    if len(tokens2) <= maxlen_A:
        tokens2 = tokens2 + ['[PAD]' for _ in range(maxlen_A - len(tokens2))]
    else:
        tokens2 = tokens2[:maxlen_A]
            
    tokens = ["[CLS]"]+tokens2+["[SEP]"]
    tokens_ids = the_tokenizer.convert_tokens_to_ids(tokens)
    tokens_ids_tensor = torch.tensor([tokens_ids])
    attn_mask = (tokens_ids_tensor != 1).long() # [PAD] => 1

    with torch.no_grad():
        # Get input embeddings (e.g., from the model's embedding layer)
        embedding_output = wte(tokens_ids_tensor)
    
    return tokens_ids_tensor, embedding_output, attn_mask

In [ ]:
import matplotlib
matplotlib.rcParams['font.family'] = 'serif' 
matplotlib.rcParams['font.serif'] = ['DejaVu Serif']

def compute_integrated_gradients(essay_id):

    input_ids, seq, attn_masks = preprocess(df, essay_id)

    seq, attn_masks = seq.cuda(), attn_masks.cuda()

    real_score = df[df["essay_id"] == essay_id].iloc[0]["test"]
    predicted_score = df[df["essay_id"] == essay_id].iloc[0]["pred"]

    # Example usage (for a single input sequence)
    seq, attn_masks = seq.cuda(), attn_masks.cuda()  # Your token IDs and attention masks

    # Compute clause attributions
    clause_attributions = compute_clause_attributions(input_ids, seq, attn_masks, max)
    clauses = clause_attributions.items()
    M = max(clause_attributions.values())

    # Define a color map that distinguishes positive and negative attributions
    def attribution_color(attribution, M=1):
        if attribution > 0:
            # Green for positive attributions
            return plt.cm.Greens(attribution/M)
        else:
            # Red for negative attributions
            return plt.cm.Reds(-attribution/M)

    # Combine all clauses into a single essay text for the left side
    essay_text = ". ".join([f"[{i+1}.] {clause[0]}" for i, clause in enumerate(clauses)])+"."

    # Create a figure with two subplots: one for the essay and one for the bar plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 8), dpi=120)

    # Left subplot: essay text with clauses enumerated
    ax1.axis('off')  # Turn off the axis for the left plot
    # Title for the left subplot
    ax1.text(.05+0.18, 1.059-0.28+0.03-0.17+0.025+0.06, 'Essay', ha='center', fontsize=12)#, fontweight='bold')

    # Display the essay text in a text box with wrapping
    ax1.text(.23, 1-0.25+0.03-0.17+0.025+0.06, essay_text, fontsize=10, verticalalignment='top', horizontalalignment='center', 
            wrap=True, bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.0))#, edgecolor="tab:olive"))

    # Display the prompt text
    ax1.text(0.05+0.18, 1.059-0.28+0.03+0.05+0.01, 'Prompt', ha='center', fontsize=12)#, fontweight='bold')
    ax1.text(0.23, 1-0.25+0.03+0.05+0.01, meta_type[str(ESSAY_SET)]["question"], fontsize=10, verticalalignment='top', horizontalalignment='center', 
            wrap=True, bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgrey', alpha=0.5, edgecolor="black"))

    # Add value boxes for real and predicted scores
    ax1.text(0.05-0.3, 1.01-0.055, 'Scores', ha='left', fontsize=12)#, fontweight='bold')
    ax1.text(0.05+0.4-0.165, 1.01-0.07, f'Real:\n{real_score:.1f}', horizontalalignment='left', fontsize=10, 
            bbox=dict(boxstyle='round,pad=0.3', facecolor='tab:gray', alpha=0.3, edgecolor="black"))
    ax1.text(0.05+0.58-0.165, 1.01-0.07, f'Predicted:\n{predicted_score:.1f}', horizontalalignment='left', fontsize=10, 
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', alpha=0.4, edgecolor="tab:green"))

    # Add score range annotation
    ax1.text(0.05-0.0, 1.01-0.07, f'Range:\n[{S_MIN}, {S_MAX}]', horizontalalignment='left', fontsize=10, 
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightblue', alpha=0.4, edgecolor="tab:blue"))


    # Right subplot: attribution scores with bars
    y_positions = []

    # Right subplot: attribution scores with bars
    for i, (clause, attribution) in enumerate(clauses):
        y_position = len(clauses) - i - 1 * 0.6  # Adjusting the y_position to space out the boxes
        ax2.barh(y_position, attribution, height=0.4, color=attribution_color(attribution, M))
        y_positions.append(y_position)
        # Label the attribution value next to the bar
        ax2.text(attribution, y_position, f'{attribution:.2f}', va='center', fontsize=10, color='black')

    # Set titles and labels for the right subplot
    ax2.set_title('Attribution Scores per Clause', fontsize=12)
    ax2.set_xlabel('Attribution Score', fontsize=10)
    # ax2.set_ylabel('Clause Number', fontsize=10)

    # Customize y-axis labels and ticks
    ax2.tick_params(axis='y', direction='out', length=6, labelright=True, labelleft=False, right=True, left=False)

    # Customize the borders of the bars
    for patch in ax2.patches:
        patch.set_edgecolor(patch._original_facecolor)  # Set the border color to black
        patch.set_linewidth(1.3)  # Set the thickness of the border
        

    # Set custom y-ticks with enumeration 1 to N
    ax2.set_yticks(y_positions)  # Set y-tick positions
    ax2.set_yticklabels([f"[{i+1}.]" for i in range(len(clauses))])  # Set y-tick labels as clause numbers

    # Set the limits and grid for the bar plot
    #ax2.set_xlim(-1, 1)  # Limit x-axis to include both positive and negative scores
    ax2.set_xlim([min(0,min(clause_attributions.values())*1.2), max(clause_attributions.values())*1.2])
    ax2.axvline(x=0, color='black', linestyle='--', alpha=0.5)  # Vertical line for 0 attribution
    ax2.grid(True, axis='x', linestyle='-', alpha=0.7)

    plt.tight_layout(pad=6)
    plt.savefig(f"interpret_{essay_id}.pdf", bbox_inches='tight', transparent=True)
    plt.show()

compute_integrated_gradients(essay_id=9878)

In [ ]:
# Function to split text into clauses based on punctuation
def split_into_terms(tokens):
    # Ensure 'text' is a string (if it's a list, join into a string)
    terms = [the_tokenizer.convert_tokens_to_string(tokens[i:i+1]).replace("#", "") for i in range(len(tokens))]
    return terms

# Define function to compute attributions for each clause
def compute_term_attributions(input_ids, seq, attn_masks):
    # Split the input sequence into clauses (by tokens)

    terms = split_into_terms(the_tokenizer.convert_ids_to_tokens(input_ids[0].cpu().numpy()))

    baseline_seq = torch.full_like(seq, the_tokenizer.pad_token_id).cuda()

    # Initialize Integrated Gradients
    ig = IntegratedGradients(forward_func)

    # Compute attributions (by tokens)
    attributions, _ = ig.attribute(inputs=seq,
                                       baselines=baseline_seq,
                                       additional_forward_args=(attn_masks,),
                                       return_convergence_delta=True)

    torch.cuda.empty_cache()  # Clear the cached memory in CUDA

    # Sum attributions across the embedding dimension to get relevance per token
    attributions = attributions.sum(dim=-1).squeeze(0)  # Sum across embedding dimensions
    attributions = attributions.detach().cpu().numpy()

    # Create a dictionary to hold total attributions by clause
    term_attributions = {}
    token_idx = 1
    # Loop over each clause and sum the attributions for the tokens within that clause
    for i, term in enumerate(terms[1:-1]):
        term_tokens = the_tokenizer.tokenize(term)
        term_length = len(term_tokens)
        term_attr = sum(attributions[token_idx:token_idx + term_length])  # Sum token attributions for the clause
        if term != "[PAD]": term_attributions[i] = (term, term_attr) 
        token_idx += term_length  # Move to the next clause

    return term_attributions

In [ ]:
import textwrap

text = """
This is a long piece of text <that> <we> <want> to wrap by words, ensuring that it doesn't split words in the middle of a line.
"""

wrapped_text = textwrap.fill(text, width=40)

print(wrapped_text)


In [ ]:
import matplotlib
from highlight_text import HighlightText, ax_text, fig_text
import textwrap


matplotlib.rcParams['font.family'] = 'serif' 
matplotlib.rcParams['font.serif'] = ['DejaVu Serif']

def compute_integrated_gradients_per_term(essay_id,move_text=0):

    input_ids, seq, attn_masks = preprocess(df, essay_id)

    seq, attn_masks = seq.cuda(), attn_masks.cuda()

    real_score = df[df["essay_id"] == essay_id].iloc[0]["test"]
    predicted_score = df[df["essay_id"] == essay_id].iloc[0]["pred"]

    # Example usage (for a single input sequence)
    seq, attn_masks = seq.cuda(), attn_masks.cuda()  # Your token IDs and attention masks

    # Compute clause attributions
    term_attributions = compute_term_attributions(input_ids, seq, attn_masks)
    term_name = {k: v[0] for k, v in term_attributions.items()}
    term_attrib = {k: v[1] for k, v in term_attributions.items()}

    M = max(term_attrib.values())

    # Define a color map that distinguishes positive and negative attributions
    def attribution_style(attribution, M=1, m=0.1):

        if attribution > m*M:
            # Green for positive attributions
            return {"color": ["black", "white"][attribution/M>0.65], "bbox": {"edgecolor": "lightblue", "facecolor": plt.cm.Blues(attribution/M), "linewidth": 0.5, "pad": 2}} 
        elif attribution < -m*M:
            # Red for negative attributions
            return {"color": ["black", "white"][-attribution/M>0.65], "bbox": {"edgecolor": "lightcoral", "facecolor": plt.cm.Reds(-attribution/M), "linewidth": 0.5, "pad": 2}} 
        else:
            return {"color": "black", "bbox": {"edgecolor": "lightgray", "facecolor": "white", "linewidth": 0, "pad": 2}} 

    # Combine all clauses into a single essay text for the left side
    essay_text = " ".join([f"<{term}>" for i, term in term_name.items()])

    # Create a figure with two subplots: one for the essay and one for the bar plot
    fig, ax1 = plt.subplots(1, 1, figsize=(6, 5), dpi=120)

    # Left subplot: essay text with clauses enumerated
    ax1.axis('off')  # Turn off the axis for the left plot
    # Title for the left subplot
    ax1.text(.05+0.18, 1.059-0.28+0.03-0.17+0.025+0.04+move_text, 'Essay', ha='center', fontsize=12)#, fontweight='bold')

    # Display the essay text in a text box with wrapping
#     ax1.text(.23, 1-0.25+0.03-0.17+0.025+0.06, essay_text, fontsize=10, verticalalignment='top', horizontalalignment='center', 
#             wrap=True, bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.0))#, edgecolor="tab:olive"))


    HighlightText(x=.23, y=1-0.25+0.03-0.17+0.025+0.04+move_text,
              s=textwrap.fill(essay_text, width=100), #'The weather is <sunny>\nYesterday it was <cloudy>', #essay_text
              va='top', ha='center',
              highlight_textprops=[attribution_style(attribution, M) for i, attribution in term_attrib.items()],
              textalign='center',
              ax=ax1)
    
    # Display the prompt text
    ax1.text(0.05+0.18, 1.059-0.28+0.03+0.05+0.01, 'Prompt', ha='center', fontsize=12)#, fontweight='bold')
    ax1.text(0.23, 1-0.25+0.03+0.05+0.01, meta_type[str(ESSAY_SET)]["question"], fontsize=10, verticalalignment='top', horizontalalignment='center', 
            wrap=True, bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgrey', alpha=0.5, edgecolor="black"))

    # Add value boxes for real and predicted scores
    ax1.text(0.05-0.3, 1.01-0.055, 'Scores', ha='left', fontsize=12)#, fontweight='bold')
    ax1.text(0.05+0.4-0.165, 1.01-0.07, f'Real:\n{real_score:.1f}', horizontalalignment='left', fontsize=10, 
            bbox=dict(boxstyle='round,pad=0.3', facecolor='tab:gray', alpha=0.3, edgecolor="black"))
    ax1.text(0.05+0.58-0.165, 1.01-0.07, f'Predicted:\n{predicted_score:.1f}', horizontalalignment='left', fontsize=10, 
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', alpha=0.4, edgecolor="tab:green"))

    # Add score range annotation
    ax1.text(0.05-0.0, 1.01-0.07, f'Range:\n[{S_MIN}, {S_MAX}]', horizontalalignment='left', fontsize=10, 
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightblue', alpha=0.4, edgecolor="tab:blue"))

    plt.tight_layout(pad=6)
    plt.savefig(f"interpret_ulra_{essay_id}.pdf", bbox_inches='tight', transparent=True)
    plt.show()

In [ ]:
for essay_id in [ 9878, 10507, 10307, 10352,  9249,  8883, 10416,  9662]:
    compute_integrated_gradients_per_term(essay_id=essay_id, move_text=-0.07)

In [ ]:
our_ig = pd.read_csv("../../nllf_method/lr_Z_C/pred/llm/w_r_sf/nllf_ef/integrated_gradients.csv", index_col=0)
our_ig

In [ ]:
essay_id = 9088
compute_integrated_gradients_per_term(essay_id=essay_id, move_text=-.15)

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.colorbar import ColorbarBase
import textwrap

In [ ]:
igs = our_ig
prompt = meta_type[str(ESSAY_SET)]["question"]
essay_id=9088
k=10
max_chars=1950

# Filter the DataFrame for the specific essay_id
df_row = igs[igs['essay_id'] == essay_id].iloc[0]

# Extract real score, predicted score, and essay text
real_score = df_row['test']
predicted_score = df_row['pred']
essay_text = df_row['essay']

# Truncate the essay text if it exceeds max_chars
if len(essay_text) > max_chars:
    essay_text = essay_text[:max_chars] + "..."

# Extract attribution scores, linguistic features (LF), and NLLF features
feature_columns = [col for col in igs.columns if col not in ['essay_id', 'test', 'pred', 'essay', 'const']]
nllf_features = [col for col in feature_columns if all(['c' in col, "(" in col, ")" in col])]  # NLLF features start with 'c'
lf_features = [col for col in feature_columns if 'ulra_' in col]  # Linguistic features (LF) start with 'ulra_'

# Select the top-k absolute feature importance
attributions = {col: df_row[col] for col in feature_columns}
sorted_attributions = sorted(attributions.items(), key=lambda x: abs(x[1]), reverse=True)[:k]

# Create two lists to store feature names and their corresponding values
features = [f[0] for f in sorted_attributions if abs(f[1])>0.005]
values = [f[1] for f in sorted_attributions if abs(f[1])>0.005]


# Classify features as NLLF or LF based on their names
colors = ['tab:orange' if f in nllf_features else 'tab:blue' for f in features]  # Blue for NLLF, Red for LF

# Plotting
fig, ax = plt.subplots(2, 4, figsize=(0.7*4*12/2, 1*9/2), dpi=120)

# Rename features if they exist in the 'rename' dictionary, otherwise keep original names

renamed_features = [textwrap.fill(rename.get(f, f), width=40) if f in nllf_features else textwrap.fill(rename.get(f, f), width=40) if f in lf_features else f for f in features]

ax[0, 3].set_axisbelow(True)
ax[0, 3].xaxis.grid(color='gray', linestyle='-')

# Barplot for feature importance
sns.barplot(x=list(range(len(renamed_features))), y=values, palette=colors, orient='v', ax=ax[0, 3], dodge=False, alpha=0.3)
# ax[1].set_xlabel('Feature Attribution Score')
# ax[0, 3].set_title(f'Higher Feature Attribution Score')#, fontweight='bold')

ax[0, 3].set_xticklabels([f"{i+1}." for i in range(len(renamed_features))], fontsize=9)#text_nllf, fontsize=9)  # Last 5 features (NLLF)


# Annotate bars with the actual values
pos_sum = sum([v for v in values if v>0])
neg_sum = sum([v for v in values if v<0])
for i, v in enumerate(values):
    ax[0, 3].text(i, -0.25, f'{ v:.2f}\n({v/pos_sum:.2f})' if v>0 else f'{ v:.2f}\n({v/neg_sum:.2f})' if v<0 else f'{ v:.2f}', color='black', ha='center', va="bottom", alpha=0.7)# , fontweight='bold')

# Customize y-axis labels and ticks
# ax[0, 1].tick_params(axis='y', direction='out', length=6, labelright=True, labelleft=False, right=True, left=False)
L, R = min(0, (min(values)-0.05)), max(0, (max(values)+0.05))
ax[0, 3].set_ylim([min(L, -R), max(-L, R)])
ax[0, 3].axhline(0, color="black", linestyle="--", alpha=0.6)


inverted_names = {i: text for i, text in enumerate(renamed_features)}

features_table = "\n".join([f"{i+1}. {textwrap.fill(text, 40)}" for i, text in inverted_names.items()])

ax[1, 3].text(0-0.1,1-0.1, features_table, verticalalignment='top', horizontalalignment='left', wrap=True,
        fontsize=11, color='black',
        bbox=dict(facecolor='lightgray', edgecolor='gray', boxstyle='round,pad=0.8', alpha=0.4),
        transform=ax[1, 3].transAxes)
ax[1, 3].axis('off')

# Customize the borders of the bars
for patch in ax[0, 3].patches:
    patch.set_edgecolor(patch._original_facecolor)  # Set the border color to black
    patch.set_linewidth(2)  # Set the thickness of the border

ax[1, 1].axis('off')

# # Add legend for NLLF and LF
# handles = [plt.Line2D([0], [0], color='tab:orange', lw=4), plt.Line2D([0], [0], color='tab:blue', lw=4)]
# labels = ['NLLF', 'EF']
# ax[0, 3].legend(handles, labels, loc="best", fontsize=9, ncol=2)
# ax[1].set_xlim([min(0, min(values)*1.4), max(0, (max(values)+0.06)*1.4)])

# fig.set_size_inches(fig.get_size_inches() * np.array([1, 1]), forward=True)

input_ids, seq, attn_masks = preprocess(df, essay_id)

seq, attn_masks = seq.cuda(), attn_masks.cuda()

real_score = df[df["essay_id"] == essay_id].iloc[0]["test"]
predicted_score = df[df["essay_id"] == essay_id].iloc[0]["pred"]

# Example usage (for a single input sequence)
seq, attn_masks = seq.cuda(), attn_masks.cuda()  # Your token IDs and attention masks

# Compute clause attributions
term_attributions = compute_term_attributions(input_ids, seq, attn_masks)
term_name = {k: v[0] for k, v in term_attributions.items()}
term_attrib = {k: v[1] for k, v in term_attributions.items()}

M = max(term_attrib.values())

# Define a color map that distinguishes positive and negative attributions
def attribution_style(attribution, M=1, m=0.05):

    if attribution > m*M:
        # Green for positive attributions
        return {"color": ["black", "white"][attribution/M>0.65], "bbox": {"edgecolor": "lightgreen", "facecolor": plt.cm.Greens(attribution/M), "linewidth": 0.5, "pad": 1}} 
    elif attribution < -m*M:
        # Red for negative attributions
        return {"color": ["black", "white"][-attribution/M>0.65], "bbox": {"edgecolor": "lightcoral", "facecolor": plt.cm.Reds(-attribution/M), "linewidth": 0.5, "pad": 1}} 
    else:
        return {"color": "black", "bbox": {"edgecolor": "lightgray", "facecolor": "white", "linewidth": 0, "pad": 1}} 

# Create two subplots: one for the essay and one for the feature importance barplot
ax[0, 1].axis('off')  # No axis for the text box
ax[0, 1].text(1, 0.7,f'Essay 2', horizontalalignment='center', fontsize=13)#, fontweight='bold')
# ax[0, 0].text(0.23, 0.695+move_text, essay_text, fontsize=10, verticalalignment='top', horizontalalignment='center', 
#             wrap=True, 
#             bbox=dict(boxstyle='round,pad=0.3',facecolor='lightyellow', alpha=0, edgecolor="tab:olive"))

ax[0, 0].text(0.35, 1.02-0.25-0.05+0.0+0.2-0.2-0.5+0.25+0.2,f'Task 4 Prompt:', horizontalalignment='center', fontsize=12)#, fontweight='bold')
ax[0, 0].text(0.35, 0.98-0.25-0.05-0.1+0.2-0.2-0.5+0.25+0.2, textwrap.fill(prompt, width=35), fontsize=12, verticalalignment='top', horizontalalignment='center', 
            wrap=True, 
            bbox=dict(boxstyle='round,pad=0.3',facecolor='lightgrey', alpha=0.5, edgecolor="black"))


# Combine all clauses into a single essay text for the left side
essay_text = " ".join([f"<{term}>" for i, term in term_name.items()])


HighlightText(x=1, y=0.5,
            s=textwrap.fill(essay_text, width=90), #'The weather is <sunny>\nYesterday it was <cloudy>', #essay_text
            va='top', ha='center',
            highlight_textprops=[attribution_style(attribution, M) for i, attribution in term_attrib.items()],
            textalign='center',
            ax=ax[0,1], fontsize=13)

# # Create a custom colormap from greens, white, and reds
# cmap = LinearSegmentedColormap.from_list('custom_cmap', [plt.cm.Reds(1.0), 'lightcoral', 'white', 'lightgreen', plt.cm.Greens(1.0)])

# # Set up normalization (for the color range)
# norm = Normalize(vmin=-1, vmax=1)

# # Customize position and size of the colorbar (within ax[0,0])
# cbar_ax = fig.add_axes([0.35, -0.1, 0.3, .05])  # [x, y, width, height] in figure coordinates

# # Add the colorbar to the defined axis
# cbar = ColorbarBase(cbar_ax, cmap=cmap, norm=norm, orientation='horizontal')

# # Set colorbar label
# cbar.set_label('Integrated Gradient Intensity', fontsize=12)
# cbar.ax.tick_params(labelsize=11)

# Add value boxes for real and predicted scores
ax[1,0].text(0.35, 1.4-0.05-1.5+0.3-0.2+0.25+0.6+0.2, 'Reference scores:', ha='center', fontsize=12)#, fontweight='bold')
# Add score range annotation

ax[1,0].text(0.11-0.1-0.02, 1.3-0.05-1.5+0.15-0.2+0.25+0.6+0.25, f'Range: [{S_MIN}, {S_MAX}]', horizontalalignment='left', fontsize=12, 
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgrey', alpha=0.4, edgecolor="black"))
        
ax[1,0].text(0.44, 1.3-0.05-1.5+0.15-0.2+0.25+0.6+0.25, f'Real: {real_score:.1f}', horizontalalignment='left', fontsize=12, 
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgrey', alpha=0.5, edgecolor="black"))

ax[1,0].text(0.35+0.02, 1.1-1.5-0.2+0.25+0.6+0.2+0.1, 'Predicted scores:', ha='center', fontsize=12)#, fontweight='bold')

ax[1,0].text(0.05+0.08+0.02-0.095, 1.0-1.5-0.15-0.2+0.25+0.6+0.25+0.1, f'ULRA: {predicted_score:.1f}', horizontalalignment='left', fontsize=12, 
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', alpha=0.4, edgecolor="tab:green"))

ax[1,0].text(0.32+0.08+0.02, 1.0-1.5-0.15-0.2+0.25+0.6+0.25+0.1, f'Our: {df_row["pred"]:.1f}', horizontalalignment='left', fontsize=12, 
        bbox=dict(boxstyle='round,pad=0.3', facecolor='tab:orange', alpha=0.3, edgecolor="tab:orange"))



ax[1, 0].axis('off')  # No axis for the text box
ax[0, 0].axis('off')  # No axis for the text box

ax[0, 2].axis('off')  # No axis for the text box
ax[1, 2].axis('off')  # No axis for the text box

# ax[0, 0].text(0.35, .95, f'Local Interpretability with\nIntegrated Gradients:', horizontalalignment='center', fontsize=12, fontweight='bold')
# ax[0, 0].text(0.35, .88, f'ULRA and Our method', horizontalalignment='center', fontsize=12)

# plt.tight_layout(pad=0)
plt.savefig(f"both_interpret_{essay_id}.pdf", bbox_inches='tight', transparent=True)
plt.show()



In [ ]:

igs = our_ig
prompt = meta_type[str(ESSAY_SET)]["question"]
essay_id=9374
k=10
max_chars=1950

# Filter the DataFrame for the specific essay_id
df_row = igs[igs['essay_id'] == essay_id].iloc[0]

# Extract real score, predicted score, and essay text
real_score = df_row['test']
predicted_score = df_row['pred']
essay_text = df_row['essay']

# Truncate the essay text if it exceeds max_chars
if len(essay_text) > max_chars:
    essay_text = essay_text[:max_chars] + "..."

# Extract attribution scores, linguistic features (LF), and NLLF features
feature_columns = [col for col in igs.columns if col not in ['essay_id', 'test', 'pred', 'essay', 'const']]
nllf_features = [col for col in feature_columns if all(['c' in col, "(" in col, ")" in col])]  # NLLF features start with 'c'
lf_features = [col for col in feature_columns if 'ulra_' in col]  # Linguistic features (LF) start with 'ulra_'

# Select the top-k absolute feature importance
attributions = {col: df_row[col] for col in feature_columns}
sorted_attributions = sorted(attributions.items(), key=lambda x: abs(x[1]), reverse=True)[:k]

# Create two lists to store feature names and their corresponding values
features = [f[0] for f in sorted_attributions if abs(f[1])>0]
values = [f[1] for f in sorted_attributions if abs(f[1])>0]


# Classify features as NLLF or LF based on their names
colors = ['tab:blue' if f in nllf_features else 'tab:orange' for f in features]  # Blue for NLLF, Red for LF

# Plotting
fig, ax = plt.subplots(2, 3, figsize=(3*12/2, 4), dpi=120)

# Rename features if they exist in the 'rename' dictionary, otherwise keep original names

ll_nllf =9

textstr_nllf = lambda ix: f"{ix}" if len(ix.split(' ')) <= ll_nllf else f"{' '.join(ix.split(' ')[:ll_nllf]).strip()}\n{' '.join(ix.split(' ')[ll_nllf:2*ll_nllf]).strip()}" if len(ix.split(' '))<=ll_nllf*2 else f"{' '.join(ix.split(' ')[:ll_nllf]).strip()}\n{' '.join(ix.split(' ')[ll_nllf:2*ll_nllf]).strip()}\n{' '.join(ix.split(' ')[2*ll_nllf:3*ll_nllf]).strip()}" if len(ix.split(' ')) <= 3*ll_nllf else f"{' '.join(ix.split(' ')[:ll_nllf]).strip()}\n{' '.join(ix.split(' ')[ll_nllf:2*ll_nllf]).strip()}\n{' '.join(ix.split(' ')[2*ll_nllf:3*ll_nllf]).strip()}\n{' '.join(ix.split(' ')[3*ll_nllf:4*ll_nllf]).strip()}" if len(ix.split(' ')) <= 4*ll_nllf else f"{' '.join(ix.split(' ')[:ll_nllf]).strip()}\n{' '.join(ix.split(' ')[ll_nllf:2*ll_nllf]).strip()}\n{' '.join(ix.split(' ')[2*ll_nllf:3*ll_nllf]).strip()}\n{' '.join(ix.split(' ')[3*ll_nllf:4*ll_nllf]).strip()}\n{' '.join(ix.split(' ')[4*ll_nllf:]).strip()}"
    
ll_ef = 10

textstr_ef = lambda ix: f"{ix}" if len(ix.split(' ')) <= ll_ef else f"{' '.join(ix.split(' ')[:ll_ef]).strip()}\n{' '.join(ix.split(' ')[ll_ef:2*ll_ef]).strip()}" if len(ix.split(' '))<=ll_ef*2 else f"{' '.join(ix.split(' ')[:ll_ef]).strip()}\n{' '.join(ix.split(' ')[ll_ef:2*ll_ef]).strip()}\n{' '.join(ix.split(' ')[2*ll_ef:3*ll_ef]).strip()}" if len(ix.split(' ')) <= 3*ll_ef else f"{' '.join(ix.split(' ')[:ll_ef]).strip()}\n{' '.join(ix.split(' ')[ll_ef:2*ll_ef]).strip()}\n{' '.join(ix.split(' ')[2*ll_ef:3*ll_ef]).strip()}\n{' '.join(ix.split(' ')[3*ll_ef:]).strip()}"


renamed_features = [textstr_nllf(rename.get(f, f)) if f in nllf_features else textstr_ef(rename.get(f, f)) if f in lf_features else f for f in features]

ax[0, 2].set_axisbelow(True)
ax[0, 2].xaxis.grid(color='gray', linestyle='-')

# Barplot for feature importance
sns.barplot(x=list(range(len(renamed_features))), y=values, palette=colors, orient='v', ax=ax[0, 2], dodge=False, alpha=0.3)
# ax[1].set_xlabel('Feature Attribution Score')
# ax[0, 2].set_title(f'Higher Feature Attribution Score')#, fontweight='bold')

reverse_inverted_names = {text: i for i, text in inverted_names.items()}

ax[0, 2].set_xticklabels([f"{reverse_inverted_names[text]+1}." for text in renamed_features], fontsize=9)#text_nllf, fontsize=9)  # Last 5 features (NLLF)


# Annotate bars with the actual values
pos_sum = sum([v for v in values if v>0])
neg_sum = sum([v for v in values if v<0])
for i, v in enumerate(values):
    ax[0, 2].text(i, v, f'{ v:.2f}\n({v/pos_sum:.2f})' if v>0 else f'{ v:.2f}\n({v/neg_sum:.2f})' if v<0 else f'{ v:.2f}', color='black', ha='center', va="top" if v<0  else "bottom", alpha=0.7)# , fontweight='bold')

# Customize y-axis labels and ticks
# ax[0, 1].tick_params(axis='y', direction='out', length=6, labelright=True, labelleft=False, right=True, left=False)
ax[0, 2].set_ylim([min(0, (min(values)-0.02)*1.2), max(0, (max(values)+0.04)*1.3)])
ax[0, 2].axhline(0, color="black", linestyle="--", alpha=0.6)


features_table = "\n".join([f"{i+1}. {text}" for i, text in enumerate(renamed_features)])

# ax[1, 1].text(0-0.26,1.5+0.04, features_table, verticalalignment='top', horizontalalignment='left', wrap=True,
#         fontsize=10, color='black',
#         bbox=dict(facecolor='lightgray', edgecolor='gray', boxstyle='round,pad=0.8', alpha=0.4),
#         transform=ax[1, 1].transAxes)

ax[1, 2].axis('off')

# Customize the borders of the bars
for patch in ax[0, 2].patches:
    patch.set_edgecolor(patch._original_facecolor)  # Set the border color to black
    patch.set_linewidth(2)  # Set the thickness of the border

ax[1, 1].axis('off')
# # Add value boxes for real and predicted scores
# ax[0].text(0.05-0.3, 1.01-0.055,f'Scores', horizontalalignment='left', fontsize=12)#, fontweight='bold')
# ax[0].text(0.05+0.4-0.165, 1.01-0.07, f'Real:\n{real_score:.1f}', horizontalalignment='left', fontsize=10, bbox=dict(boxstyle='round,pad=0.3',facecolor='tab:gray', alpha=0.3, edgecolor="black"))
# ax[0].text(0.05+0.58-0.165, 1.01-0.07, f'Predicted:\n{predicted_score:.1f}', horizontalalignment='left', fontsize=10, bbox=dict(boxstyle='round,pad=0.3',facecolor='lightgreen', alpha=0.4, edgecolor="tab:green"))

# # Add score range annotation
# ax[0].text(0.05-0.0, 1.01-0.07, f'Range:\n[{S_MIN}, {S_MAX}]', horizontalalignment='left', fontsize=10, bbox=dict(boxstyle='round,pad=0.3',facecolor='lightblue', alpha=0.4, edgecolor="tab:blue"))

# # Add legend for NLLF and LF
handles = [plt.Line2D([0], [0], color='tab:blue', lw=4), plt.Line2D([0], [0], color='tab:orange', lw=4)]
labels = ['NLLF', 'Linguistic\nFeatures']
# ax[0, 2].legend(handles, labels, title='Feature Types', loc="upper right", fontsize=10, title_fontsize=10)
# ax[1].set_xlim([min(0, min(values)*1.4), max(0, (max(values)+0.06)*1.4)])

# fig.set_size_inches(fig.get_size_inches() * np.array([1, 1]), forward=True)

input_ids, seq, attn_masks = preprocess(df, essay_id)

seq, attn_masks = seq.cuda(), attn_masks.cuda()

real_score = df[df["essay_id"] == essay_id].iloc[0]["test"]
predicted_score = df[df["essay_id"] == essay_id].iloc[0]["pred"]

# Example usage (for a single input sequence)
seq, attn_masks = seq.cuda(), attn_masks.cuda()  # Your token IDs and attention masks

# Compute clause attributions
term_attributions = compute_term_attributions(input_ids, seq, attn_masks)
term_name = {k: v[0] for k, v in term_attributions.items()}
term_attrib = {k: v[1] for k, v in term_attributions.items()}

M = max(term_attrib.values())

# Define a color map that distinguishes positive and negative attributions
def attribution_style(attribution, M=1, m=0.05):

    if attribution > m*M:
        # Green for positive attributions
        return {"color": ["black", "white"][attribution/M>0.65], "bbox": {"edgecolor": "lightgreen", "facecolor": plt.cm.Greens(attribution/M), "linewidth": 0.5, "pad": 1}} 
    elif attribution < -m*M:
        # Red for negative attributions
        return {"color": ["black", "white"][-attribution/M>0.65], "bbox": {"edgecolor": "lightcoral", "facecolor": plt.cm.Reds(-attribution/M), "linewidth": 0.5, "pad": 1}} 
    else:
        return {"color": "black", "bbox": {"edgecolor": "lightgray", "facecolor": "white", "linewidth": 0, "pad": 1}} 

# Create two subplots: one for the essay and one for the feature importance barplot
ax[0, 1].axis('off')  # No axis for the text box
# ax[0, 1].text(0.35, 1.02,f'Essay', horizontalalignment='center', fontsize=12)#, fontweight='bold')
# ax[0, 0].text(0.23, 0.695+move_text, essay_text, fontsize=10, verticalalignment='top', horizontalalignment='center', 
#             wrap=True, 
#             bbox=dict(boxstyle='round,pad=0.3',facecolor='lightyellow', alpha=0, edgecolor="tab:olive"))

# ax[0, 0].text(0.35, 1.02-0.25-0.05+0.2,f'Task 4 Prompt:', horizontalalignment='center', fontsize=12)#, fontweight='bold')
# ax[0, 0].text(0.35, 0.98-0.25-0.05+0.2, textwrap.fill(prompt, width=50), fontsize=10, verticalalignment='top', horizontalalignment='center', 
#             wrap=True, 
#             bbox=dict(boxstyle='round,pad=0.3',facecolor='lightgrey', alpha=0.5, edgecolor="black"))


# Combine all clauses into a single essay text for the left side
essay_text = " ".join([f"<{term}>" for i, term in term_name.items()])


HighlightText(x=0.35, y=1,
            s=textwrap.fill(essay_text, width=100), #'The weather is <sunny>\nYesterday it was <cloudy>', #essay_text
            va='top', ha='center',
            highlight_textprops=[attribution_style(attribution, M) for i, attribution in term_attrib.items()],
            textalign='center',
            ax=ax[0,1])

# Create a custom colormap from greens, white, and reds
# cmap = LinearSegmentedColormap.from_list('custom_cmap', [plt.cm.Reds(1.0), 'lightcoral', 'white', 'lightgreen', plt.cm.Greens(1.0)])

# Set up normalization (for the color range)
# norm = Normalize(vmin=-1, vmax=1)

# Customize position and size of the colorbar (within ax[0,0])
# cbar_ax = fig.add_axes([0.332, 0.4, 0.3, .02])  # [x, y, width, height] in figure coordinates

# Add the colorbar to the defined axis
# cbar = ColorbarBase(cbar_ax, cmap=cmap, norm=norm, orientation='horizontal')

# Set colorbar label
# cbar.set_label('Integrated Gradient Intensity', fontsize=12)

# Add value boxes for real and predicted scores
# ax[1,0].text(0.35, 1.4-0.05+0.3, 'Reference scores:', ha='center', fontsize=12)#, fontweight='bold')
# # Add score range annotation

# ax[1,0].text(0.11, 1.3-0.05+0.3, f'Range: [{S_MIN}, {S_MAX}]', horizontalalignment='left', fontsize=12, 
#         bbox=dict(boxstyle='round,pad=0.3', facecolor='lightblue', alpha=0.4, edgecolor="tab:blue"))
        
# ax[1,0].text(0.44, 1.3-0.05+0.3, f'Real: {real_score:.1f}', horizontalalignment='left', fontsize=12, 
#         bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgrey', alpha=0.5, edgecolor="black"))

ax[1,0].text(0.35+0.02, 1.1+0.4+0.42, 'Predicted scores:', ha='center', fontsize=12)#, fontweight='bold')

ax[1,0].text(0.05+0.08+0.02, 1.0+0.3+0.42, f'ULRA: {predicted_score:.1f}', horizontalalignment='left', fontsize=12, 
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', alpha=0.4, edgecolor="tab:green"))

ax[1,0].text(0.32+0.08+0.02, 1.0+0.3+0.42, f'Our: {df_row["pred"]:.1f}', horizontalalignment='left', fontsize=12, 
        bbox=dict(boxstyle='round,pad=0.3', facecolor='tab:orange', alpha=0.3, edgecolor="tab:orange"))


ax[1, 0].axis('off')  # No axis for the text box
ax[0, 0].axis('off')  # No axis for the text box

# ax[0, 0].text(0.35, .95, f'Local Interpretability with\nIntegrated Gradients:', horizontalalignment='center', fontsize=12, fontweight='bold')
# ax[0, 0].text(0.35, .88, f'ULRA and Our method', horizontalalignment='center', fontsize=12)

# plt.tight_layout(pad=0)
plt.savefig(f"both_interpret_{essay_id}.pdf", bbox_inches='tight', transparent=True)
plt.show()

